In [1]:
import numpy as np
from np_struct import ldarray
from dateutil import relativedelta as rdt    
import datetime as dt

## Labeled Arrays

`ldarray` is a simple interface similar to Xarray that supports indexing by coordinates. Coordinates can by integer, float
or string types.

In [2]:
coords = dict(a=[1, 2], b=['col1', 'col2', 'col3'])
ld = ldarray([[10, 11, 12],[13, 14, 15]], coords=coords, dtype=np.float64)
ld

ldarray([[10, 11, 12],
         [13, 14, 15]])
Coordinates: (2, 3)
  a: [1, 2]
  b: ['col1', 'col2', 'col3']

### Indexing

Normal indexing works the same as standard numpy arrays, including advanced indexing,

In [3]:
ld[:, 2] = 3
ld

ldarray([[10, 11,  3],
         [13, 14,  3]])
Coordinates: (2, 3)
  a: [1, 2]
  b: ['col1', 'col2', 'col3']

In [ ]:
# advanced indexing with numpy arrays as an index
ld[:, np.array([0, 2])]

ldarray([[10,  3],
         [13,  3]])
Coordinates: (2, 2)
  a: [1 2]
  b: ['col1' 'col3']

To index by coordinate value, use `.sel(...)`. Coordinate indexing can be done with slices, endpoint is inclusive.

In [5]:
ld.sel(b="col1")

ldarray([10, 13])
Coordinates: (2,)
  a: [1 2]

In [6]:
# select every other column
ld.sel(a=1, b=slice('col1', 'col3', 2))

ldarray([10,  3])
Coordinates: (2,)
  b: ['col1' 'col3']

To set by coordinate value, use a dictionary indexer,

In [7]:
ld[dict(a=2)] = [1, 2, 3]
ld

ldarray([[10, 11,  3],
         [ 1,  2,  3]])
Coordinates: (2, 3)
  a: [1, 2]
  b: ['col1', 'col2', 'col3']

The coordinates will be dropped if the shape is changed by a math operation. In this case the user is responsible
for casting the array back into a `ldarray` if needed. 

In [8]:
sum_array = ldarray(ld.sum(axis=0), coords=dict(b=['col1', 'col2', 'col3']))
sum_array

ldarray([11, 13,  6])
Coordinates: (3,)
  b: ['col1', 'col2', 'col3']

### Indexing Tolerance

Indexing tolerance can be set on a per-dimension basis, 

In [9]:
coords = dict(a=[1.2, 2.4, 3.1], b=[4, 5], idx_precision=dict(a=1e-2))
ld2 = ldarray([[10, 11],[12, 13],[14, 15]], coords=coords)
ld2

ldarray([[10, 11],
         [12, 13],
         [14, 15]])
Coordinates: (3, 2)
  a: [1.2 2.4 3.1]
  b: [4, 5]

In [10]:
# this will raise an error because the selection is further than 1e-2 away from any coordinate value
try:
    ld2[dict(a=1.21)]
except IndexError as m:
    print(m)

Coordinate 1.21 is outside precision given for dimension key a.


In [11]:
# but this will work
ld2[dict(a=1.201)]

ldarray([10, 11])
Coordinates: (2,)
  b: [4 5]

### Date Coordinates

`ldarray` also supports datetime objects as coordinate values.

In [12]:
start = dt.datetime.today()
dates = [start + rdt.relativedelta(months=i) for i in range(12)]

data = ldarray(np.arange(0, 12), coords=dict(time=dates))

data

ldarray([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])
Coordinates: (12,)
  time: ['2026-03-23T00:00' '2026-04-23T00:00' ... '2027-01-23T00:00'
   '2027-02-23T00:00']

In [13]:
data.sel(
    time=slice(
        start + rdt.relativedelta(months=6), 
        start + rdt.relativedelta(months=9)
    )
)

ldarray([6, 7, 8, 9])
Coordinates: (4,)
  time: ['2026-09-23T00:00' '2026-10-23T00:00' '2026-11-23T00:00'
   '2026-12-23T00:00']

### Saving as .npy Files

Labeled arrays can be saved to disk in the standard `.npy` format and still retain their coordinates.

In [14]:
data.save("test.npy")

In [15]:
ldarray.load("test.npy", allow_pickle=True)

ldarray([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])
Coordinates: (12,)
  time: ['2026-03-23T00:00' '2026-04-23T00:00' ... '2027-01-23T00:00'
   '2027-02-23T00:00']